# RivalRadar Agentic Pipeline (Agent 1-4)

Single notebook pipeline:
- Agent 1: scrape + parse competitor pages
- Agent 2: OpenAI vulnerability analysis
- Agent 3: OpenAI pricing-change prediction
- Agent 4: OpenAI action recommendation

Goal: run one end-to-end pipeline in one notebook.

## 1. Imports, OpenAI Key, and Shared Configuration

In [58]:
import asyncio
import json
import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from openai import OpenAI

from competitor_targets import COMPETITOR_TARGETS
from scrapers.output_parser import parse_pricing_page, save_structured

# Crawlee is optional for live scraping; fallback mode can still run without it.
try:
    from scrapers.crawlee_scraper import CrawleeScraper
    CRAWLEE_AVAILABLE = True
except ModuleNotFoundError:
    CrawleeScraper = None
    CRAWLEE_AVAILABLE = False

OUTPUT_DIR = Path("scraper_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── API Key ───────────────────────────────────────────────────────────────────
# Load from .env.example in the same directory as this notebook.
def _load_env_file(path: Path) -> dict:
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                env[k.strip()] = v.strip()
    return env

_env = _load_env_file(Path(".env")) or _load_env_file(Path(".env.example"))
OPENAI_API_KEY = _env.get("OPENAI_API_KEY", "").strip()

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found or empty in .env / .env.example")
print("✓ API key loaded from .env.example")
# ─────────────────────────────────────────────────────────────────────────────

OPENAI_MODEL = _env.get("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = _env.get("OPENAI_BASE_URL", "").strip() or None

RUN_LIVE_SCRAPE = CRAWLEE_AVAILABLE
VERBOSE = True
MAX_RETRIES = 1

print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Competitors configured: {len(COMPETITOR_TARGETS)}")
print(f"Crawlee available: {CRAWLEE_AVAILABLE}")
print(f"Live scrape mode: {RUN_LIVE_SCRAPE}")
print(f"OpenAI model: {OPENAI_MODEL}")

if not CRAWLEE_AVAILABLE:
    print("Install missing deps in your venv: pip install -r requirements.txt")


def get_openai_client() -> OpenAI:
    return OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)


def chat_json(client: OpenAI, system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict[str, Any]:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("OpenAI returned empty content")
    return json.loads(content)

✓ API key loaded from .env.example
Output dir: /Users/hrithikkannankrishnan/Desktop/Innovation Challenge /DBA5115-rivalradar/rivalradar/scraper_output
Competitors configured: 5
Crawlee available: True
Live scrape mode: True
OpenAI model: llama-3.3-70b-versatile


## 2. Define First Agent Interface and Implementation

In [59]:
class Agent1Collector:
    """Collect competitor pricing pages and emit structured competitor profiles."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.scraper = CrawleeScraper() if CRAWLEE_AVAILABLE else None

    def build_targets(self) -> list[dict[str, str]]:
        return [
            {
                "url": target["pricing_url"],
                "page_type": "pricing",
                "name": target["name"],
            }
            for target in COMPETITOR_TARGETS
        ]

    async def collect(self) -> dict[str, Any]:
        if self.scraper is None:
            raise ModuleNotFoundError(
                "crawlee is not installed. Install deps with: pip install -r requirements.txt"
            )

        targets = self.build_targets()
        scrape_results = await self.scraper.scrape_pages(targets)

        profiles: list[dict[str, Any]] = []
        for result in scrape_results:
            profile = parse_pricing_page(
                raw_text=result.get("raw_text", ""),
                company=result.get("name", "Unknown"),
                url=result.get("url", ""),
                scraped_at=result.get("scraped_at", ""),
            )
            save_structured(profile, self.output_dir)
            profiles.append(profile)

        return {
            "agent": "agent1_collector",
            "target_count": len(targets),
            "scrape_result_count": len(scrape_results),
            "structured_profiles": profiles,
        }


def load_existing_structured_profiles(output_dir: Path) -> list[dict[str, Any]]:
    profiles: list[dict[str, Any]] = []
    for path in sorted(output_dir.glob("*_structured.json")):
        with open(path, encoding="utf-8") as f:
            profiles.append(json.load(f))
    return profiles

## 3. Define Agent 2 (OpenAI Vulnerability Analysis)

In [60]:
@dataclass
class ScoreComponent:
    name: str
    weight: float
    score: float
    weighted_score: float
    reasoning: str


@dataclass
class VulnerabilityResult:
    company: str
    vulnerability_score: float
    risk_level: str
    confidence: float
    reasoning_summary: str
    detailed_reasoning: list[str]
    decision_trace: list[str]
    component_breakdown: list[ScoreComponent]
    signals: dict[str, int]
    metrics: dict[str, Any]
    scraped_at: str
    peer_rank: int | None
    peer_percentile: float | None


class Agent2CompetitiveAnalyzer:
    """Analyze competitor profiles with OpenAI and return explainable vulnerability results."""

    component_weights = {
        "network_effects": 0.15,
        "switching_costs": 0.15,
        "economies_of_scale": 0.10,
        "proprietary_technology": 0.15,
        "brand_strength": 0.10,
        "data_moat": 0.15,
        "integration_lock_in": 0.10,
        "regulatory_barriers": 0.10,
    }

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _analyze_one(self, profile: dict[str, Any]) -> VulnerabilityResult:
        prompt = (
            "You are a venture capital analyst assessing portfolio company defensibility. Return strict JSON only. "
            "Score this company's competitive moat across 8 dimensions to produce a portfolio risk assessment.\n"
            "Required keys: vulnerability_score, confidence, risk_level, reasoning_summary, "
            "detailed_reasoning, decision_trace, component_breakdown, signals, metrics.\n"
            "component_breakdown must be a list of 8 objects, each with keys: name (string), score (float 0.0-1.0), reasoning (string). "
            "Use these name values exactly: network_effects, switching_costs, economies_of_scale, proprietary_technology, brand_strength, data_moat, integration_lock_in, regulatory_barriers.\n"
            "A score of 1.0 means strongest possible moat for that dimension. vulnerability_score = 1 - average weighted score.\n"
            "signals must include ai_momentum, enterprise_readiness, integration_strength as integers.\n"
            f"Profile: {json.dumps(profile, ensure_ascii=False)}"
        )
        payload = chat_json(
            self.client,
            "Return only valid JSON, no markdown.",
            prompt,
            temperature=0.1,
        )

        raw_component_names = [
            item.get("name", "")
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        ]
        if VERBOSE:
            print(f"[Agent2] Raw component names from LLM: {raw_component_names}")

        components_by_name = {
            item.get("name", "").lower().strip().replace(" ", "_"): item
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        }

        components: list[ScoreComponent] = []
        for name, weight in self.component_weights.items():
            raw = components_by_name.get(name, {})
            score = max(0.0, min(1.0, float(raw.get("score", 0.0))))
            components.append(
                ScoreComponent(
                    name=name,
                    weight=weight,
                    score=round(score, 3),
                    weighted_score=round(score * weight, 4),
                    reasoning=str(raw.get("reasoning", "No reasoning provided.")).strip(),
                )
            )

        vulnerability_score = max(0.0, min(1.0, float(payload.get("vulnerability_score", sum(c.weighted_score for c in components)))))
        confidence = max(0.0, min(1.0, float(payload.get("confidence", 0.6))))
        risk_level = str(payload.get("risk_level", "medium")).lower().strip()
        if risk_level not in {"low", "medium", "high", "critical"}:
            risk_level = "high" if vulnerability_score >= 0.65 else ("medium" if vulnerability_score >= 0.45 else "low")

        signals_raw = payload.get("signals", {}) if isinstance(payload.get("signals"), dict) else {}
        signals = {
            "ai_momentum": int(signals_raw.get("ai_momentum", 0)),
            "enterprise_readiness": int(signals_raw.get("enterprise_readiness", 0)),
            "integration_strength": int(signals_raw.get("integration_strength", 0)),
        }

        metrics_raw = payload.get("metrics", {}) if isinstance(payload.get("metrics"), dict) else {}
        metrics = {
            "plan_count": int(metrics_raw.get("plan_count", len(profile.get("plans", [])))),
            "price_range": metrics_raw.get("price_range", profile.get("pricing_summary", {}).get("price_range", "N/A")),
            "raw_char_count": int(metrics_raw.get("raw_char_count", profile.get("raw_char_count", 0))),
            "avg_features_per_plan": float(metrics_raw.get("avg_features_per_plan", 0.0)),
            "unique_feature_count": int(metrics_raw.get("unique_feature_count", 0)),
            "numeric_price_count": int(metrics_raw.get("numeric_price_count", 0)),
        }

        detailed_reasoning = payload.get("detailed_reasoning", [])
        if not isinstance(detailed_reasoning, list):
            detailed_reasoning = [str(detailed_reasoning)]

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        return VulnerabilityResult(
            company=profile.get("company", "Unknown"),
            vulnerability_score=round(vulnerability_score, 3),
            risk_level=risk_level,
            confidence=round(confidence, 3),
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            detailed_reasoning=[str(x) for x in detailed_reasoning],
            decision_trace=[str(x) for x in decision_trace],
            component_breakdown=components,
            signals=signals,
            metrics=metrics,
            scraped_at=profile.get("scraped_at", datetime.now(timezone.utc).isoformat()),
            peer_rank=None,
            peer_percentile=None,
        )

    def analyze(self, agent1_output: dict[str, Any]) -> dict[str, Any]:
        profiles = agent1_output["structured_profiles"]
        results = [self._analyze_one(profile) for profile in profiles]
        results.sort(key=lambda r: r.vulnerability_score, reverse=True)

        total = len(results)
        for idx, result in enumerate(results, start=1):
            result.peer_rank = idx
            result.peer_percentile = 1.0 if total == 1 else round(1 - ((idx - 1) / (total - 1)), 3)
            result.reasoning_summary = (
                f"Ranked #{result.peer_rank}/{total} (peer_percentile={result.peer_percentile:.3f}). "
                f"{result.reasoning_summary}"
            )

        report_path = self.output_dir / "vulnerability_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_vulnerability_v1",
                    "results": [asdict(r) for r in results],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent2_analyzer",
            "vulnerability_results": results,
            "report_path": str(report_path),
            "analyzed_count": len(results),
        }

## 4. Define Agent 3 (Pricing Predictor)

In [61]:
@dataclass
class ImpactForecast:
    company: str
    revenue_at_risk_pct: float
    time_to_impact: str
    risk_level: str
    reasoning_summary: str
    impact_drivers: dict[str, float]
    decision_trace: list[str]
    evidence: dict[str, Any]
    generated_at: str


class Agent3PricingPredictor:
    """Predict competitor pricing-change likelihood and timeline using OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _predict_one(self, vuln: VulnerabilityResult, profile: dict[str, Any]) -> ImpactForecast:
        context = {
            "company": vuln.company,
            "vulnerability_score": vuln.vulnerability_score,
            "risk_level": vuln.risk_level,
            "confidence": vuln.confidence,
            "signals": vuln.signals,
            "metrics": vuln.metrics,
            "profile": profile,
        }
        prompt = (
            "You are a VC analyst forecasting financial impact on a portfolio company from competitive threats. Return strict JSON only.\n"
            "Required keys: revenue_at_risk_pct, time_to_impact, risk_level, reasoning_summary, impact_drivers, decision_trace, evidence.\n"
            "revenue_at_risk_pct is a float 0.0-1.0 representing share of revenue threatened.\n"
            "time_to_impact must be one of: 0-3 months, 3-6 months, 6-18 months, 18+ months.\n"
            "impact_drivers must include: moat_weakness, competitive_pressure, market_timing, execution_risk, customer_stickiness, analysis_confidence with [0,1] floats.\n"
            f"Context: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, "Return only valid JSON.", prompt, temperature=0.15)
        probability = max(0.0, min(1.0, float(payload.get("revenue_at_risk_pct", 0.0))))
        risk = str(payload.get("risk_level", "low")).lower().strip()
        if risk not in {"low", "medium", "high"}:
            risk = "high" if probability >= 0.75 else ("medium" if probability >= 0.5 else "low")

        timeline = str(payload.get("time_to_impact", "3+ months")).strip()
        if timeline not in {"0-3 months", "3-6 months", "6-18 months", "18+ months"}:
            timeline = "0-3 months" if probability >= 0.8 else ("3-6 months" if probability >= 0.65 else ("6-18 months" if probability >= 0.5 else "18+ months"))

        raw_drivers = payload.get("impact_drivers", {}) if isinstance(payload.get("impact_drivers"), dict) else {}
        drivers = {
            "moat_weakness": round(max(0.0, min(1.0, float(raw_drivers.get("moat_weakness", vuln.vulnerability_score)))), 3),
            "competitive_pressure": round(max(0.0, min(1.0, float(raw_drivers.get("competitive_pressure", 0.0)))), 3),
            "market_timing": round(max(0.0, min(1.0, float(raw_drivers.get("market_timing", 0.0)))), 3),
            "execution_risk": round(max(0.0, min(1.0, float(raw_drivers.get("execution_risk", 0.0)))), 3),
            "customer_stickiness": round(max(0.0, min(1.0, float(raw_drivers.get("customer_stickiness", 0.0)))), 3),
            "analysis_confidence": round(max(0.0, min(1.0, float(raw_drivers.get("analysis_confidence", vuln.confidence)))), 3),
        }

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("price_range", vuln.metrics.get("price_range", "N/A"))
        evidence.setdefault("peer_rank", vuln.peer_rank)

        return ImpactForecast(
            company=vuln.company,
            revenue_at_risk_pct=round(probability, 3),
            time_to_impact=timeline,
            risk_level=risk,
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            impact_drivers=drivers,
            decision_trace=[str(x) for x in decision_trace],
            evidence=evidence,
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def predict(self, agent1_output: dict[str, Any], agent2_output: dict[str, Any]) -> dict[str, Any]:
        profiles = {p.get("company", "Unknown"): p for p in agent1_output["structured_profiles"]}
        vulnerability_results = agent2_output["vulnerability_results"]

        predictions = [
            self._predict_one(vuln, profiles.get(vuln.company, {}))
            for vuln in vulnerability_results
        ]
        predictions.sort(key=lambda p: p.revenue_at_risk_pct, reverse=True)

        report_path = self.output_dir / "impact_forecasts_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_pricing_prediction_v1",
                    "results": [asdict(p) for p in predictions],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent3_pricing",
            "impact_forecasts": predictions,
            "report_path": str(report_path),
            "prediction_count": len(predictions),
        }

## 5. Define Agent 4 (Action Planner)

In [62]:
@dataclass
class ActionRecommendation:
    company: str
    priority: str
    recommendation_type: str
    owner: str
    due_window: str
    action_title: str
    action_detail: str
    rationale: str
    evidence: dict[str, Any]
    decision_trace: list[str]
    generated_at: str


class Agent4ActionPlanner:
    """Generate VC-facing action recommendations from Agent 2 + Agent 3 with OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _recommend_one(self, vuln: VulnerabilityResult, pred: PricingPrediction) -> ActionRecommendation:
        context = {
            "company": vuln.company,
            "vulnerability_score": vuln.vulnerability_score,
            "vulnerability_risk": vuln.risk_level,
            "vulnerability_summary": vuln.reasoning_summary,
            "pricing_change_probability": pred.revenue_at_risk_pct,
            "pricing_timeline": pred.time_to_impact,
            "pricing_risk": pred.risk_level,
            "pricing_summary": pred.reasoning_summary,
        }
        prompt = (
            "You are a senior VC partner generating board-level strategic actions. Return strict JSON only.\n"
            "Required keys: priority, recommendation_type, owner, due_window, action_title, action_detail, rationale, evidence, decision_trace.\n"
            "priority must be one of: P0, P1, P2, P3.\n"
            "recommendation_type must be one of: acquisition_target, product_pivot, pricing_defense, partnership_acceleration, monitor_only.\n"
            "owner must be a board or fund-level role (e.g. Lead Partner, Portfolio Ops, Investment Committee).\n"
            "Frame action_detail as a board-meeting directive, not a product feature request.\n"
            f"Context: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, "Return only valid JSON.", prompt, temperature=0.2)
        priority = str(payload.get("priority", "P2")).upper().strip()
        if priority not in {"P0", "P1", "P2", "P3"}:
            priority = "P2"

        rec_type = str(payload.get("recommendation_type", "monitor_only")).lower().strip()
        if rec_type not in {"acquisition_target", "product_pivot", "pricing_defense", "partnership_acceleration", "monitor_only"}:
            rec_type = "monitor_only"

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("vulnerability_score", vuln.vulnerability_score)
        evidence.setdefault("pricing_change_probability", pred.revenue_at_risk_pct)

        return ActionRecommendation(
            company=vuln.company,
            priority=priority,
            recommendation_type=rec_type,
            owner=str(payload.get("owner", "Portfolio Operations Lead")).strip(),
            due_window=str(payload.get("due_window", "14 days")).strip(),
            action_title=str(payload.get("action_title", f"Monitor {vuln.company}")).strip(),
            action_detail=str(payload.get("action_detail", "No detail provided.")).strip(),
            rationale=str(payload.get("rationale", "No rationale provided.")).strip(),
            evidence=evidence,
            decision_trace=[str(x) for x in decision_trace],
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def recommend(self, agent2_output: dict[str, Any], agent3_output: dict[str, Any]) -> dict[str, Any]:
        vulnerability_results = agent2_output["vulnerability_results"]
        predictions = {p.company: p for p in agent3_output["impact_forecasts"]}

        recommendations: list[ActionRecommendation] = []
        for vuln in vulnerability_results:
            pred = predictions.get(vuln.company)
            if pred is None:
                continue
            recommendations.append(self._recommend_one(vuln, pred))

        priority_order = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}
        recommendations.sort(key=lambda r: (priority_order.get(r.priority, 9), r.company))

        report_path = self.output_dir / "action_recommendations_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_action_recommendation_v1",
                    "results": [asdict(r) for r in recommendations],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent4_actions",
            "action_recommendations": recommendations,
            "report_path": str(report_path),
            "recommendation_count": len(recommendations),
        }

## 6. Compose Multi-Agent Pipeline (Agents 1-4)

In [63]:
async def run_agent_pipeline(run_live_scrape: bool = True) -> dict[str, Any]:
    start = time.perf_counter()

    client = get_openai_client()

    agent1 = Agent1Collector(OUTPUT_DIR)
    agent2 = Agent2CompetitiveAnalyzer(OUTPUT_DIR, client)
    agent3 = Agent3PricingPredictor(OUTPUT_DIR, client)
    agent4 = Agent4ActionPlanner(OUTPUT_DIR, client)

    agent1_output: dict[str, Any] | None = None

    if run_live_scrape:
        for attempt in range(MAX_RETRIES + 1):
            try:
                if VERBOSE:
                    print(f"[Agent1] Live scrape attempt {attempt + 1}/{MAX_RETRIES + 1}")
                agent1_output = await agent1.collect()
                if VERBOSE:
                    print(f"[Agent1] Scraped {len(agent1_output.get('structured_profiles', []))} profiles.")
                break
            except Exception as exc:
                print(f"[Agent1] Attempt failed: {exc}")
                if attempt == MAX_RETRIES:
                    print("[Agent1] Falling back to existing structured JSON files.")

    # Fall back if scrape was skipped, failed, or returned no profiles
    if agent1_output is None or len(agent1_output.get("structured_profiles", [])) == 0:
        profiles = load_existing_structured_profiles(OUTPUT_DIR)
        if VERBOSE:
            print(f"[Agent1] Loaded {len(profiles)} existing structured profiles from disk.")
        agent1_output = {
            "agent": "agent1_collector_fallback",
            "target_count": len(COMPETITOR_TARGETS),
            "scrape_result_count": 0,
            "structured_profiles": profiles,
        }

    agent2_output = agent2.analyze(agent1_output)
    agent3_output = agent3.predict(agent1_output, agent2_output)
    agent4_output = agent4.recommend(agent2_output, agent3_output)

    elapsed = time.perf_counter() - start
    if VERBOSE:
        print(f"[Pipeline] Completed in {elapsed:.2f}s")

    return {
        "elapsed_seconds": elapsed,
        "agent1_output": agent1_output,
        "agent2_output": agent2_output,
        "agent3_output": agent3_output,
        "agent4_output": agent4_output,
    }

## 7. Run End-to-End Example (Agents 1-4)

In [64]:
pipeline_result = await run_agent_pipeline(run_live_scrape=RUN_LIVE_SCRAPE)

print("Agent 1 output keys:", list(pipeline_result["agent1_output"].keys()))
print("Agent 2 output keys:", list(pipeline_result["agent2_output"].keys()))
print("Agent 3 output keys:", list(pipeline_result["agent3_output"].keys()))
print("Agent 4 output keys:", list(pipeline_result["agent4_output"].keys()))

vulnerability_results = pipeline_result["agent2_output"]["vulnerability_results"]
pricing_predictions = pipeline_result["agent3_output"]["impact_forecasts"]
action_recommendations = pipeline_result["agent4_output"]["action_recommendations"]

print("Agent 2 report:", pipeline_result["agent2_output"]["report_path"])
print("Agent 3 report:", pipeline_result["agent3_output"]["report_path"])
print("Agent 4 report:", pipeline_result["agent4_output"]["report_path"])

[scrapers.crawlee_scraper] INFO  No APIFY_TOKEN found — running without proxy.  Set APIFY_TOKEN in .env for residential proxy rotation.
[crawlee.crawlers._playwright._playwright_crawler] INFO  Current request statistics:
┌───────────────────────────────┬──────┐
│ requests_finished             │ 0    │
│ requests_failed               │ 0    │
│ retry_histogram               │ [0]  │
│ request_avg_failed_duration   │ None │
│ request_avg_finished_duration │ None │
│ requests_finished_per_minute  │ 0    │
│ requests_failed_per_minute    │ 0    │
│ request_total_duration        │ 0s   │
│ requests_total                │ 0    │
│ crawler_runtime               │ 0s   │
└───────────────────────────────┴──────┘
[crawlee.crawlers._playwright._playwright_crawler] INFO  Crawled 0/5 pages, 0 failed requests, desired concurrency 3.


[Agent1] Live scrape attempt 1/2


[crawlee._autoscaling.autoscaled_pool] INFO  current_concurrency = 0; desired_concurrency = 3; cpu = 0.0; mem = 0.0; event_loop = 0.0; client_info = 0.0
[crawlee._autoscaling.autoscaled_pool] INFO  Waiting for remaining tasks to finish
[crawlee.crawlers._playwright._playwright_crawler] INFO  Final request statistics:
┌───────────────────────────────┬─────────┐
│ requests_finished             │ 0       │
│ requests_failed               │ 0       │
│ retry_histogram               │ [0]     │
│ request_avg_failed_duration   │ None    │
│ request_avg_finished_duration │ None    │
│ requests_finished_per_minute  │ 0       │
│ requests_failed_per_minute    │ 0       │
│ request_total_duration        │ 0s      │
│ requests_total                │ 0       │
│ crawler_runtime               │ 532.8ms │
└───────────────────────────────┴─────────┘


[Agent1] Scraped 0 profiles.
[Agent1] Loaded 5 existing structured profiles from disk.
[Agent2] Raw component names from LLM: ['network_effects', 'switching_costs', 'economies_of_scale', 'proprietary_technology', 'brand_strength', 'data_moat', 'integration_lock_in', 'regulatory_barriers']
[Agent2] Raw component names from LLM: ['network_effects', 'switching_costs', 'economies_of_scale', 'proprietary_technology', 'brand_strength', 'data_moat', 'integration_lock_in', 'regulatory_barriers']
[Agent2] Raw component names from LLM: ['network_effects', 'switching_costs', 'economies_of_scale', 'proprietary_technology', 'brand_strength', 'data_moat', 'integration_lock_in', 'regulatory_barriers']
[Agent2] Raw component names from LLM: ['network_effects', 'switching_costs', 'economies_of_scale', 'proprietary_technology', 'brand_strength', 'data_moat', 'integration_lock_in', 'regulatory_barriers']
[Agent2] Raw component names from LLM: ['network_effects', 'switching_costs', 'economies_of_scale', '

In [65]:
vulnerability_rows = []
for result in vulnerability_results:
    top_component = max(result.component_breakdown, key=lambda c: c.weighted_score)
    vulnerability_rows.append(
        {
            "company": result.company,
            "rank": result.peer_rank,
            "vulnerability_score": result.vulnerability_score,
            "risk_level": result.risk_level,
            "confidence": result.confidence,
            "top_driver": top_component.name,
            "signal_hits": sum(result.signals.values()),
            "reasoning_summary": result.reasoning_summary,
        }
    )

pricing_rows = [
    {
        "company": p.company,
        "revenue_at_risk_pct": p.revenue_at_risk_pct,
        "pricing_risk": p.risk_level,
        "time_to_impact": p.time_to_impact,
        "top_driver": max(p.impact_drivers.items(), key=lambda item: item[1])[0],
        "reasoning_summary": p.reasoning_summary,
    }
    for p in pricing_predictions
]

action_rows = [
    {
        "company": a.company,
        "priority": a.priority,
        "recommendation_type": a.recommendation_type,
        "owner": a.owner,
        "due_window": a.due_window,
        "action_title": a.action_title,
        "rationale": a.rationale,
    }
    for a in action_recommendations
]

vulnerability_df = pd.DataFrame(vulnerability_rows)
if not vulnerability_df.empty:
    vulnerability_df = vulnerability_df.sort_values("rank")

pricing_df = pd.DataFrame(pricing_rows)
if not pricing_df.empty:
    pricing_df = pricing_df.sort_values("revenue_at_risk_pct", ascending=False)

actions_df = pd.DataFrame(action_rows)
if not actions_df.empty:
    actions_df = actions_df.sort_values("priority")

pd.set_option("display.max_colwidth", 220)
print("PORTFOLIO RISK ASSESSMENT")
display(vulnerability_df)
print("IMPACT FORECAST")
display(pricing_df)
print("VC ACTION RECOMMENDATIONS")
display(actions_df)

PORTFOLIO RISK ASSESSMENT


,company,rank,vulnerability_score,risk_level,confidence,top_driver,signal_hits,reasoning_summary
0,ClickUp,1,0.42,low,0.80,network_effects,21,"Ranked #1/5 (peer_percentile=1.000). ClickUp has a moderate competitive moat due to its strong network effects, brand strength, and data moat. However, it faces intense competition in the project management space."
1,HubSpot,2,0.42,low,0.85,network_effects,24,"Ranked #2/5 (peer_percentile=0.750). HubSpot has a strong brand and network effects, but its proprietary technology and regulatory barriers are not as robust."
2,Linear,3,0.42,low,0.80,network_effects,21,"Ranked #3/5 (peer_percentile=0.500). Linear has a moderate competitive moat, driven by its strong network effects, brand strength, and data moat. However, the company's vulnerability score is elevated due to relative..."
3,Notion,4,0.42,low,0.80,network_effects,21,"Ranked #4/5 (peer_percentile=0.250). Notion has a strong brand and network effects, but its proprietary technology and regulatory barriers are relatively weak."
4,Stripe,5,0.38,low,0.85,network_effects,21,"Ranked #5/5 (peer_percentile=0.000). Stripe has a strong competitive moat due to its network effects, brand strength, and economies of scale, but is vulnerable in areas such as proprietary technology and regulatory b..."


IMPACT FORECAST


,company,revenue_at_risk_pct,pricing_risk,time_to_impact,top_driver,reasoning_summary
0,ClickUp,0.25,low,6-18 months,customer_stickiness,"ClickUp's vulnerability score is 0.42, indicating a moderate level of vulnerability. However, the company's strong AI momentum and integration strength mitigate this risk."
1,HubSpot,0.25,low,6-18 months,analysis_confidence,"Competitive threats are moderate due to HubSpot's established market presence, but vulnerability score indicates some weaknesses"
2,Linear,0.25,low,6-18 months,customer_stickiness,"Linear's vulnerability score of 0.42 and low risk level indicate a moderate threat to revenue. However, the company's strong ai momentum and enterprise readiness signals mitigate this risk."
3,Notion,0.25,low,6-18 months,customer_stickiness,"Notion's vulnerability score is 0.42, indicating a relatively low risk. However, the company's moat weakness and competitive pressure from other note-taking and collaboration tools pose a threat to its revenue."
4,Stripe,0.15,low,6-18 months,analysis_confidence,"Stripe's vulnerability score is 0.38, indicating a relatively low risk. However, the company's reliance on transaction-based pricing and lack of fixed plan tiers may make it vulnerable to competitive pressure."


VC ACTION RECOMMENDATIONS


,company,priority,recommendation_type,owner,due_window,action_title,rationale
0,ClickUp,P1,monitor_only,Lead Partner,Q2 2024,ClickUp Competitive Positioning Review,"ClickUp's moderate competitive moat and low vulnerability risk suggest a stable market position, but intense competition in the project management space warrants regular monitoring to identify potential risks or oppo..."
1,HubSpot,P1,monitor_only,Investment Committee,Q1-Q2,HubSpot Vulnerability and Pricing Risk Assessment,"HubSpot's low vulnerability risk and moderate competitive threats indicate a stable market position, but ongoing monitoring is necessary to ensure the company's continued success and address potential weaknesses"
2,Notion,P1,monitor_only,Investment Committee,Q1-Q2,Notion Vulnerability and Pricing Risk Assessment,"Notion's low vulnerability score and pricing risk indicate a stable position, but ongoing monitoring is necessary to address potential threats from competitors and changes in the market"
3,Stripe,P1,monitor_only,Investment Committee,Q2 2024,Stripe Competitive Positioning Review,Stripe's low vulnerability score and strong competitive moat warrant continued monitoring to ensure the company's market position remains stable
4,Linear,P2,monitor_only,Portfolio Ops,Q2-Q3,Linear Vulnerability and Pricing Risk Review,"Linear's moderate competitive moat and low pricing risk suggest a stable position, but ongoing monitoring is necessary to ensure the company remains proactive in addressing potential vulnerabilities and capitalizing ..."


## 8. Validation and Debug Output (Agents 1-4)

In [66]:
agent1_output = pipeline_result["agent1_output"]
agent2_output = pipeline_result["agent2_output"]
agent3_output = pipeline_result["agent3_output"]
agent4_output = pipeline_result["agent4_output"]

assert "structured_profiles" in agent1_output, "Agent 1 output missing structured_profiles"
assert isinstance(agent1_output["structured_profiles"], list), "structured_profiles must be a list"
assert len(agent1_output["structured_profiles"]) > 0, "No structured profiles available"

assert "vulnerability_results" in agent2_output, "Agent 2 output missing vulnerability_results"
assert isinstance(agent2_output["vulnerability_results"], list), "vulnerability_results must be a list"
assert len(agent2_output["vulnerability_results"]) > 0, "Agent 2 returned no vulnerability results"

assert "impact_forecasts" in agent3_output, "Agent 3 output missing impact_forecasts"
assert isinstance(agent3_output["impact_forecasts"], list), "pricing_predictions must be a list"
assert len(agent3_output["impact_forecasts"]) > 0, "Agent 3 returned no impact forecasts"

assert "action_recommendations" in agent4_output, "Agent 4 output missing action_recommendations"
assert isinstance(agent4_output["action_recommendations"], list), "action_recommendations must be a list"
assert len(agent4_output["action_recommendations"]) > 0, "Agent 4 returned no recommendations"

sample_profile = agent1_output["structured_profiles"][0]
required_profile_keys = {"company", "pricing_summary", "raw_char_count"}
assert required_profile_keys.issubset(sample_profile.keys()), "Profile schema mismatch"

if VERBOSE:
    print("[Validation] Agents 1-4 outputs passed schema checks.")
    print("[Debug] Structured profiles:", len(agent1_output["structured_profiles"]))
    print("[Debug] Vulnerability results:", len(agent2_output["vulnerability_results"]))
    print("[Debug] Pricing predictions:", len(agent3_output["impact_forecasts"]))
    print("[Debug] Action recommendations:", len(agent4_output["action_recommendations"]))
    print("[Debug] Pipeline elapsed seconds:", round(pipeline_result["elapsed_seconds"], 2))

[Validation] Agents 1-4 outputs passed schema checks.
[Debug] Structured profiles: 5
[Debug] Vulnerability results: 5
[Debug] Pricing predictions: 5
[Debug] Action recommendations: 5
[Debug] Pipeline elapsed seconds: 25.07


## 9. RivalRadar Interactive Dashboard

Plotly-based dashboard displaying pipeline outputs: Portfolio Heatmap, Component Breakdown, Pricing Predictions, and Action Recommendations.

In [67]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Extract data ──────────────────────────────────────────────────────────────
vuln_results  = pipeline_result["agent2_output"]["vulnerability_results"]
price_preds   = pipeline_result["agent3_output"]["impact_forecasts"]
action_recs   = pipeline_result["agent4_output"]["action_recommendations"]

RISK_COLOR = {"low": "#27ae60", "medium": "#f39c12", "high": "#e74c3c", "critical": "#8e44ad"}
PRIORITY_COLOR = {"P0": "#c0392b", "P1": "#e67e22", "P2": "#2980b9", "P3": "#7f8c8d"}

# ── 1. Portfolio Heatmap ──────────────────────────────────────────────────────
companies     = [r.company for r in vuln_results]
vuln_scores   = [r.vulnerability_score for r in vuln_results]
risk_levels   = [r.risk_level.upper() for r in vuln_results]
confidences   = [r.confidence for r in vuln_results]
peer_ranks    = [str(r.peer_rank) if r.peer_rank is not None else "N/A" for r in vuln_results]
cell_colors   = [[RISK_COLOR.get(r.risk_level, "#bdc3c7") for r in vuln_results]]

fig_heatmap = go.Figure(data=[go.Table(
    columnwidth=[180, 100, 80, 90, 80],
    header=dict(
        values=["<b>Company</b>", "<b>Vuln Score</b>", "<b>Risk</b>", "<b>Confidence</b>", "<b>Peer Rank</b>"],
        fill_color="#2c3e50", font=dict(color="white", size=13), align="center",
        height=36
    ),
    cells=dict(
        values=[
            companies,
            [f"{s:.2f}" for s in vuln_scores],
            risk_levels,
            [f"{c:.0%}" for c in confidences],
            peer_ranks,
        ],
        fill_color=[
            ["#ecf0f1"] * len(companies),
            cell_colors[0],
            cell_colors[0],
            ["#ecf0f1"] * len(companies),
            ["#ecf0f1"] * len(companies),
        ],
        font=dict(size=12),
        align="center",
        height=30
    )
)])
fig_heatmap.update_layout(
    title=dict(text="<b>Portfolio Heatmap — Vulnerability Overview</b>", font=dict(size=16)),
    margin=dict(t=50, b=10, l=10, r=10),
    height=max(200, 60 + len(companies) * 34)
)
fig_heatmap.show()

# ── 3. Pricing Prediction Summary ─────────────────────────────────────────────
pp_companies   = [p.company for p in price_preds]
pp_prob        = [p.revenue_at_risk_pct for p in price_preds]
pp_timeline    = [p.time_to_impact for p in price_preds]
pp_risk        = [p.risk_level.upper() for p in price_preds]
pp_top_driver  = [max(p.impact_drivers, key=p.impact_drivers.get).replace("_", " ").title() for p in price_preds]
pp_risk_colors = [RISK_COLOR.get(p.risk_level, "#bdc3c7") for p in price_preds]

fig_pricing = go.Figure(data=[go.Table(
    columnwidth=[180, 120, 110, 80, 160],
    header=dict(
        values=["<b>Company</b>", "<b>Revenue at Risk</b>", "<b>Time to Impact</b>", "<b>Risk</b>", "<b>Top Driver</b>"],
        fill_color="#2c3e50", font=dict(color="white", size=13), align="center", height=36
    ),
    cells=dict(
        values=[
            pp_companies,
            [f"{p:.0%}" for p in pp_prob],
            pp_timeline,
            pp_risk,
            pp_top_driver,
        ],
        fill_color=[
            ["#ecf0f1"] * len(pp_companies),
            pp_risk_colors,
            ["#ecf0f1"] * len(pp_companies),
            pp_risk_colors,
            ["#ecf0f1"] * len(pp_companies),
        ],
        font=dict(size=12),
        align="center",
        height=30
    )
)])
fig_pricing.update_layout(
    title=dict(text="<b>Impact Forecast</b>", font=dict(size=16)),
    margin=dict(t=50, b=10, l=10, r=10),
    height=max(200, 60 + len(pp_companies) * 34)
)
fig_pricing.show()

# ── 4. Action Recommendations ─────────────────────────────────────────────────
ar_priorities = [r.priority for r in action_recs]
ar_companies  = [r.company for r in action_recs]
ar_titles     = [r.action_title for r in action_recs]
ar_owners     = [r.owner for r in action_recs]
ar_windows    = [r.due_window for r in action_recs]
ar_rationale  = [r.rationale[:80] + ("…" if len(r.rationale) > 80 else "") for r in action_recs]
ar_p_colors   = [PRIORITY_COLOR.get(r.priority, "#95a5a6") for r in action_recs]

fig_actions = go.Figure(data=[go.Table(
    columnwidth=[60, 160, 200, 150, 90, 250],
    header=dict(
        values=["<b>Priority</b>", "<b>Company</b>", "<b>Action</b>", "<b>Owner</b>", "<b>Due Window</b>", "<b>Rationale</b>"],
        fill_color="#2c3e50", font=dict(color="white", size=13), align="center", height=36
    ),
    cells=dict(
        values=[ar_priorities, ar_companies, ar_titles, ar_owners, ar_windows, ar_rationale],
        fill_color=[
            ar_p_colors,
            ["#ecf0f1"] * len(action_recs),
            ["#ecf0f1"] * len(action_recs),
            ["#ecf0f1"] * len(action_recs),
            ["#ecf0f1"] * len(action_recs),
            ["#ecf0f1"] * len(action_recs),
        ],
        font=dict(size=12),
        align=["center", "left", "left", "left", "center", "left"],
        height=36
    )
)])
fig_actions.update_layout(
    title=dict(text="<b>VC Action Recommendations — Prioritised (P0→P3)</b>", font=dict(size=16)),
    margin=dict(t=50, b=10, l=10, r=10),
    height=max(200, 60 + len(action_recs) * 40)
)
fig_actions.show()

import plotly.io as pio
import os

_out_path = os.path.join(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd(), "rivalradar_dashboard.html")

_heatmap_html  = pio.to_html(fig_heatmap, full_html=False, include_plotlyjs=False)
_pricing_html  = pio.to_html(fig_pricing, full_html=False, include_plotlyjs=False)
_actions_html  = pio.to_html(fig_actions, full_html=False, include_plotlyjs=False)

_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>RivalRadar Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f4f6f9; margin: 0; padding: 20px; }}
    h1 {{ color: #2c3e50; text-align: center; margin-bottom: 8px; }}
    .subtitle {{ text-align: center; color: #7f8c8d; margin-bottom: 30px; font-size: 14px; }}
    .section {{ background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); padding: 20px; margin-bottom: 24px; }}
  </style>
</head>
<body>
  <h1>RivalRadar — Competitive Intelligence Dashboard</h1>
  <p class="subtitle">Pipeline output &middot; Auto-generated</p>
  <div class="section">{_heatmap_html}</div>
  <div class="section">{_pricing_html}</div>
  <div class="section">{_actions_html}</div>
</body>
</html>"""

with open(_out_path, "w", encoding="utf-8") as _f:
    _f.write(_html)

print(f"✓ Dashboard saved to: {_out_path}")
print("Open rivalradar_dashboard.html in your browser.")


✓ Dashboard saved to: /Users/hrithikkannankrishnan/Desktop/Innovation Challenge /DBA5115-rivalradar/rivalradar/rivalradar_dashboard.html
Open rivalradar_dashboard.html in your browser.
